In [ ]:
"""
Perceptron (MLP) Hyperparameter Tuning with Optuna
====================================================
Searches for the best combination of:
  - Learning rate
  - Hidden layer sizes (width of each layer)
  - Dropout rates per layer
  - Batch size
  - Optimizer (Adam vs AdamW)
  - Number of hidden layers (depth)

After tuning, the best values are printed ready to paste into train_perceptron.py

Run:
    pip install optuna
    python tune_perceptron.py
"""

import os
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ── Config ────────────────────────────────────────────────────────────────────
DATASET_DIR = "barcode_dataset_with_fail"
IMG_SIZE    = 64
N_TRIALS    = 25      # number of hyperparameter combinations to try
TUNE_EPOCHS = 8       # epochs per trial — enough to see if it's learning
NUM_WORKERS = 0
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT_SIZE  = IMG_SIZE * IMG_SIZE * 3   # 12,288

print("=" * 60)
print("  MLP Hyperparameter Tuning — Optuna")
print("=" * 60)
print(f"  Device   : {DEVICE}")
print(f"  Trials   : {N_TRIALS}")
print(f"  Epochs   : {TUNE_EPOCHS} per trial")

# ── Fixed transforms ──────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])


# ── Dynamic MLP that Optuna can reshape ───────────────────────────────────────

class DynamicMLP(nn.Module):
    """
    MLP whose architecture is controlled by Optuna trial parameters.
    Optuna will try different numbers of layers, layer sizes, and dropouts.
    """
    def __init__(self, input_size, num_classes, layer_sizes, dropouts):
        super().__init__()
        self.flatten = nn.Flatten()

        layers = []
        in_features = input_size
        for out_features, dropout in zip(layer_sizes, dropouts):
            layers.append(nn.Linear(in_features, out_features))
            layers.append(nn.BatchNorm1d(out_features))   # helps MLP converge
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_features = out_features

        layers.append(nn.Linear(in_features, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(self.flatten(x))


# ── Optuna objective function ─────────────────────────────────────────────────

def objective(trial):
    """
    Called once per trial. Optuna picks parameters, we train a mini-model,
    return the best val accuracy. Optuna tries to maximise this.
    """

    # ── Parameters Optuna searches ────────────────────────────────────────────
    lr           = trial.suggest_float("lr",         1e-4, 1e-2, log=True)
    batch_size   = trial.suggest_categorical("batch_size",   [32, 64, 128])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW"])
    n_layers     = trial.suggest_int("n_layers", 2, 4)    # 2, 3, or 4 hidden layers
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

    # Each hidden layer gets its own size and dropout chosen independently
    layer_sizes = []
    dropouts    = []
    for i in range(n_layers):
        # Layer sizes decrease as we go deeper (pyramid shape)
        size    = trial.suggest_categorical(f"layer_{i}_size",    [256, 512, 1024])
        dropout = trial.suggest_float(      f"layer_{i}_dropout",  0.1, 0.5)
        layer_sizes.append(size)
        dropouts.append(dropout)

    print(f"\n  Trial {trial.number}: lr={lr:.5f}, batch={batch_size}, "
          f"opt={optimizer_name}, layers={layer_sizes}, "
          f"dropouts=[{', '.join(f'{d:.2f}' for d in dropouts)}]")

    # ── DataLoaders ───────────────────────────────────────────────────────────
    train_ds = datasets.ImageFolder(
        os.path.join(DATASET_DIR, "train"), transform=train_transform)
    val_ds   = datasets.ImageFolder(
        os.path.join(DATASET_DIR, "validation"), transform=eval_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_ds, batch_size=batch_size,
                              shuffle=False, num_workers=NUM_WORKERS)

    num_classes = len(train_ds.classes)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = DynamicMLP(INPUT_SIZE, num_classes, layer_sizes, dropouts).to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr,
                               weight_decay=weight_decay)
    else:
        optimizer = optim.AdamW(model.parameters(), lr=lr,
                                weight_decay=weight_decay)

    # Cosine annealing works better than ReduceLROnPlateau for short trials
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TUNE_EPOCHS)

    # ── Training ──────────────────────────────────────────────────────────────
    best_val_acc = 0.0

    for epoch in range(TUNE_EPOCHS):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Validate
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                preds    = model(images).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)

        val_acc = correct / total
        best_val_acc = max(best_val_acc, val_acc)

        # Report intermediate value — Optuna can prune bad trials early
        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_acc


# ── Run the study ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    study = optuna.create_study(
        direction="maximize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5,
                                           n_warmup_steps=3),
    )
    study.optimize(objective, n_trials=N_TRIALS)

    # ── Print best results ────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("  BEST HYPERPARAMETERS FOUND")
    print("="*60)
    best = study.best_params
    for key, val in best.items():
        print(f"  {key:30s}: {val}")
    print(f"\n  Best val accuracy : {study.best_value*100:.2f}%")

    # ── Generate copy-paste code ──────────────────────────────────────────────
    print("\n" + "="*60)
    print("  Copy these into train_perceptron.py:")
    print("="*60)

    n_layers = best["n_layers"]
    sizes    = [best[f"layer_{i}_size"]    for i in range(n_layers)]
    drops    = [best[f"layer_{i}_dropout"] for i in range(n_layers)]

    print(f"""
LEARNING_RATE = {best['lr']:.5f}
BATCH_SIZE    = {best['batch_size']}
OPTIMIZER     = "{best['optimizer']}"
WEIGHT_DECAY  = {best['weight_decay']:.5f}

# In BarcodeMLP — replace the self.network Sequential with:
layers = []
in_features = INPUT_SIZE""")

    for i, (size, drop) in enumerate(zip(sizes, drops)):
        print(f"""
# Layer {i+1}
layers.append(nn.Linear(in_features, {size}))
layers.append(nn.BatchNorm1d({size}))
layers.append(nn.ReLU())
layers.append(nn.Dropout({drop:.2f}))
in_features = {size}""")

    print("""
layers.append(nn.Linear(in_features, num_classes))
self.network = nn.Sequential(*layers)
""")

    # ── Top 5 trials summary ──────────────────────────────────────────────────
    print("="*60)
    print("  Top 5 trials:")
    print("="*60)
    sorted_trials = sorted(study.trials,
                           key=lambda t: t.value or 0,
                           reverse=True)
    for i, t in enumerate(sorted_trials[:5]):
        if t.value is not None:
            n = t.params.get("n_layers", "?")
            sz = [t.params.get(f"layer_{j}_size", "?") for j in range(n)]
            print(f"  #{i+1}  val={t.value*100:.2f}%  "
                  f"lr={t.params.get('lr', 0):.5f}  "
                  f"layers={sz}  "
                  f"opt={t.params.get('optimizer', '?')}")

[I 2026-04-04 01:15:02,272] A new study created in memory with name: no-name-f11b562b-3ad0-4c44-89a2-6832477095f8


  MLP Hyperparameter Tuning — Optuna
  Device   : cpu
  Trials   : 25
  Epochs   : 8 per trial

  Trial 0: lr=0.00043, batch=128, opt=Adam, layers=[1024, 512, 1024, 256], dropouts=[0.17, 0.23, 0.35, 0.12]


[I 2026-04-04 01:19:25,295] Trial 0 finished with value: 0.7588888888888888 and parameters: {'lr': 0.0004327186771894075, 'batch_size': 128, 'optimizer': 'Adam', 'n_layers': 4, 'weight_decay': 0.0026497718440018103, 'layer_0_size': 1024, 'layer_0_dropout': 0.1729052912473156, 'layer_1_size': 512, 'layer_1_dropout': 0.2300673550129455, 'layer_2_size': 1024, 'layer_2_dropout': 0.35036377721267487, 'layer_3_size': 256, 'layer_3_dropout': 0.11682266450492468}. Best is trial 0 with value: 0.7588888888888888.



  Trial 1: lr=0.00012, batch=128, opt=AdamW, layers=[512, 256, 512], dropouts=[0.38, 0.24, 0.35]


[I 2026-04-04 01:21:57,939] Trial 1 finished with value: 0.7488888888888889 and parameters: {'lr': 0.00011510892563324775, 'batch_size': 128, 'optimizer': 'AdamW', 'n_layers': 3, 'weight_decay': 0.006297774756727811, 'layer_0_size': 512, 'layer_0_dropout': 0.3839930937270648, 'layer_1_size': 256, 'layer_1_dropout': 0.23723284857588944, 'layer_2_size': 512, 'layer_2_dropout': 0.3530818930759224}. Best is trial 0 with value: 0.7588888888888888.


...


  Trial 8: lr=0.00152, batch=64, opt=Adam, layers=[1024, 512, 1024, 256], dropouts=[0.12, 0.11, 0.40, 0.21]


[I 2026-04-04 01:39:42,114] Trial 8 pruned. 



  Trial 9: lr=0.00078, batch=64, opt=AdamW, layers=[1024, 512, 256], dropouts=[0.46, 0.31, 0.12]


[I 2026-04-04 01:42:49,677] Trial 9 finished with value: 0.7544444444444445 and parameters: {'lr': 0.0007784510481239922, 'batch_size': 64, 'optimizer': 'AdamW', 'n_layers': 3, 'weight_decay': 0.0002631562338941644, 'layer_0_size': 1024, 'layer_0_dropout': 0.4632408117934387, 'layer_1_size': 512, 'layer_1_dropout': 0.3079064685477838, 'layer_2_size': 256, 'layer_2_dropout': 0.11878491058846063}. Best is trial 6 with value: 0.7655555555555555.



  Trial 10: lr=0.00948, batch=64, opt=AdamW, layers=[1024, 1024], dropouts=[0.24, 0.10]
